# SATO: Straighten Any 3D Tubular Object
### Colon Straightening from CT Colonography

---

This notebook demonstrates the complete **SATO pipeline** applied to a CT colonography dataset.
It follows the method published in:

> Zhou et al., *SATO: Straighten Any 3D Tubular Object*, **IEEE TMI**, 2025.
> GitHub: https://github.com/Yanfeng-Zhou/SATO

### Pipeline overview

```
DICOM series
     │
     ▼
Step 1 ─ Load & convert to NIfTI (SimpleITK)
     │
     ▼
Step 2 ─ Colon segmentation  (TotalSegmentator)
     │
     ▼
Step 3 ─ Centerline extraction (3-D skeletonize + BFS)
     │
     ▼
Step 4 ─ SATO straightening  (Zhou's Swept Frame)
     │
     ▼
  Straightened CT volume  +  Straightened mask
```

**Learning objectives**
1. Understand what *curved planar reformation* (CPR) is and why it is useful.
2. Understand the mathematical foundation of **Zhou's Swept Frame** and how it avoids the limitations of Frenet frames.
3. Run each step of the pipeline and inspect intermediate results.
4. Critically assess reproducibility: what the repository provides vs. what the paper claims.

---
## 0. Setup

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk

# make sure our local modules are importable
ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

from pipeline import (
    dicom_to_nifti, run_totalsegmentator,
    compute_centerline, straighten_volume,
)
from visualize import (
    plot_ct_slices, plot_mask_overlay,
    plot_centerline_on_slices, plot_centerline_3d,
    plot_straightened, plot_radius_profile,
    plot_pipeline_summary,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

# ── Paths ──────────────────────────────────────────────────────────────────
DICOM_DIR = str(ROOT / "data/manifest-1776479825092/CT COLONOGRAPHY"
                       "/1.3.6.1.4.1.9328.50.4.0518"
                       "/01-01-2000-1-CT COLONOGRAPHY-83932"
                       "/2.000000-SUPINE-83936")
OUT       = ROOT / "output"
OUT.mkdir(exist_ok=True)

NIFTI_PATH    = str(OUT / "ct.nii.gz")
SEG_DIR       = str(OUT / "segmentation")
MASK_PATH     = str(OUT / "segmentation" / "colon.nii.gz")
CL_PATH       = str(OUT / "centerline.npy")
STRAIGHT_IMG  = str(OUT / "straightened_image.tif")
STRAIGHT_SEG  = str(OUT / "straightened_mask.tif")

print("Setup complete.")

---
## Step 1 — Load DICOM and inspect the CT volume

The CT data is a **CT Colonography** scan: the patient's colon is insufflated with
air before scanning, which creates strong contrast between the air-filled lumen,
the colon wall, and surrounding tissue.

We read the DICOM series with **SimpleITK**, which handles the slice ordering and
spacing metadata automatically, and save as a NIfTI file for downstream tools.

In [ ]:
if not os.path.exists(NIFTI_PATH):
    img_itk = dicom_to_nifti(DICOM_DIR, NIFTI_PATH)
else:
    img_itk = sitk.ReadImage(NIFTI_PATH)
    print(f"Loaded existing NIfTI: {NIFTI_PATH}")

img_arr = sitk.GetArrayFromImage(img_itk)  # shape (Z, Y, X), HU values
spacing = img_itk.GetSpacing()             # (x_mm, y_mm, z_mm) — ITK convention
sp_zyx  = (spacing[2], spacing[1], spacing[0])  # reorder to (Z,Y,X) for numpy

print(f"Array shape  : {img_arr.shape}  (Z, Y, X)")
print(f"Voxel spacing: z={sp_zyx[0]:.3f} mm, y={sp_zyx[1]:.3f} mm, x={sp_zyx[2]:.3f} mm")
print(f"HU range     : [{img_arr.min()}, {img_arr.max()}]")
print(f"Volume size  : {img_arr.shape[0]*sp_zyx[0]:.0f} × "
      f"{img_arr.shape[1]*sp_zyx[1]:.0f} × "
      f"{img_arr.shape[2]*sp_zyx[2]:.0f} mm")

In [ ]:
# Soft-tissue window: WL=40 HU, WW=400 HU
fig = plot_ct_slices(img_arr, spacing=sp_zyx, wl=40, ww=400,
                     title="CT Colonography — soft-tissue window")
plt.show()

In [ ]:
# Lung/air window — reveals the air-filled colon lumen clearly
fig = plot_ct_slices(img_arr, spacing=sp_zyx, wl=-500, ww=1500,
                     title="CT Colonography — lung/air window (reveals colon lumen)")
plt.show()

---
## Step 2 — Colon Segmentation with TotalSegmentator

[TotalSegmentator](https://github.com/wasserth/totalsegmentator) is a nnU-Net–based
model trained to segment 104 anatomical structures from CT images.  
We use the `--fast` mode (3 mm model) for speed; the full model gives better accuracy.

> **Runtime**: ~5–15 minutes on CPU.  
> If the mask already exists in `output/segmentation/colon.nii.gz` this cell skips segmentation.

In [ ]:
run_totalsegmentator(NIFTI_PATH, SEG_DIR, fast=True)

mask_itk = sitk.ReadImage(MASK_PATH)
mask_arr = sitk.GetArrayFromImage(mask_itk).astype(np.uint8)

print(f"Mask shape       : {mask_arr.shape}")
print(f"Colon voxels     : {(mask_arr > 0).sum():,}")
print(f"Colon volume     : {(mask_arr > 0).sum() * np.prod(sp_zyx) / 1000:.1f} cm³")

In [ ]:
fig = plot_mask_overlay(img_arr, mask_arr, spacing=sp_zyx,
                        title="TotalSegmentator colon mask")
plt.show()

---
## Step 3 — Centerline Extraction

### 3.1 What is a centerline?

The **centerline** of a tubular object is a 1-D curve $\gamma(s)$ that runs through
the centre of the tube from one end to the other, where $s$ is the arc-length parameter:

$$\gamma(s) = \bigl(x(s),\, y(s),\, z(s)\bigr), \quad s \in [0, L]$$

A good centerline must:
- Pass through the lumen centre at every cross-section
- Be **smooth** (differentiable everywhere)
- Have **no spurious branches** (the colon is a single tube)

### 3.2 Algorithm: 3-D skeletonization + longest-path BFS

1. **Largest connected component** — keep only the main colon body (removes small
   detached mask fragments).

2. **Euclidean distance transform** — computes the distance from every mask voxel
   to the nearest background voxel. This gives the **local tube radius** at each point:
   $$r(\mathbf{p}) = \min_{\mathbf{q} \notin \text{mask}} \|\mathbf{p} - \mathbf{q}\|$$

3. **3-D Skeletonization (Lee 1994)** — iteratively erodes the binary mask, removing
   surface voxels that do not change the topology. The result is a 1-voxel-thin
   skeleton that preserves the topology of the original shape.

4. **Graph construction** — each skeleton voxel is a node; edges connect 26-neighbour
   voxel pairs.

5. **Double BFS (finding the diameter of a tree)**:
   - BFS from an arbitrary node → finds the farthest endpoint $e_1$
   - BFS from $e_1$ → finds the globally farthest endpoint $e_2$
   - The path $e_1 \to e_2$ is the **longest path** (colon main trunk)

6. **Per-point radius** — evaluated from the distance transform at each skeleton voxel
   on the main path.

The result is saved as a `.npy` file in SATO's required format.

In [ ]:
if not os.path.exists(CL_PATH):
    cl = compute_centerline(mask_arr)
    np.save(CL_PATH, np.array(cl, dtype=object), allow_pickle=True)
else:
    cl = np.load(CL_PATH, allow_pickle=True).tolist()
    print(f"Loaded existing centerline: {CL_PATH}")

n_pts = len(cl["point"])
pts   = np.array([p["coordinate"] for p in cl["point"]])
radii = np.array([p["width"]      for p in cl["point"]])

print(f"Centerline points : {n_pts}")
print(f"Arc length        : {cl['edge_length']:.1f} voxels  "
      f"= {cl['edge_length'] * sp_zyx[0]:.0f} mm  (approx)")
print(f"Radius range      : {radii.min():.1f} – {radii.max():.1f} voxels")
print(f"Start [Z,Y,X]     : {cl['start_coordinate']}")
print(f"End   [Z,Y,X]     : {cl['end_coordinate']}")

In [ ]:
fig = plot_centerline_on_slices(img_arr, mask_arr, cl, spacing=sp_zyx,
                                title="Colon centerline — MIP projections")
plt.show()

In [ ]:
fig = plot_centerline_3d(mask_arr, cl, spacing=sp_zyx)
plt.show()

In [ ]:
fig = plot_radius_profile(cl, spacing=sp_zyx)
plt.show()

---
## Background: Zhou's Swept Frame

This is the mathematical core of SATO. Before running Step 4, it is worth
understanding *why* a new swept frame was needed.

### Why not use the Frenet frame?

The classical **Frenet frame** defines the local coordinate system at each point
$\gamma(i)$ using the first and second derivatives of the centerline:

$$\vec{z}_i = \frac{\gamma'(i)}{|\gamma'(i)|}\qquad \text{(tangent)}
\qquad
\vec{x}_i = \frac{\gamma''(i)}{|\gamma''(i)|}\qquad \text{(normal)}
\qquad
\vec{y}_i = \vec{z}_i \times \vec{x}_i\qquad \text{(binormal)}$$

**Problems with Frenet:**
- Undefined when $\gamma''(i) = 0$ (straight segments, planar arcs with inflection points)
- The frame can flip abruptly at inflection points, causing **rotation bias** in the
  straightened result
- Requires computing the second derivative, which is expensive and numerically unstable

### Zhou's Swept Frame (rotation-based)

The key insight: instead of computing $\vec{x}_i$ from the second derivative,
**propagate it from the previous frame by rotating around the intersection line**
of adjacent cross-sections.

#### Initialisation (frame at $i = 0$)

The tangent vector is:
$$\vec{z}_0 = \gamma'(0) = \bigl(x'(0),\, y'(0),\, z'(0)\bigr)^\top$$

A perpendicular initial normal is chosen analytically:
$$\vec{x}_0 = \begin{cases}
(0,\;-z'(0),\;y'(0))^\top & \text{if } x'(0) = 0 \\
(-z'(0),\;0,\;x'(0))^\top & \text{if } y'(0) = 0 \\
(-y'(0),\;x'(0),\;0)^\top & \text{otherwise}
\end{cases}
\qquad
\vec{y}_0 = \vec{z}_0 \times \vec{x}_0$$

All three vectors are then normalised.

#### Frame update (Rodrigues' Rotation Formula)

Given the normalised tangents at adjacent points $\vec{z}_{i-1}$ and $\vec{z}_i$,
the two cross-sections $p_{i-1}$ and $p_i$ intersect along a line with unit direction:

$$\vec{a}_i = \frac{\vec{z}_{i-1} \times \vec{z}_i}{|\vec{z}_{i-1} \times \vec{z}_i|}
\tag{rotation axis}$$

The angle between the two planes (equivalently, between the two tangents) is:

$$\theta_i = \cos^{-1}\!\left(\frac{\vec{z}_{i-1} \cdot \vec{z}_i}
{|\vec{z}_{i-1}|\,|\vec{z}_i|}\right)
\tag{rotation angle}$$

**Rodrigues' Rotation Formula (RRF)** rotates a vector $\vec{v}$ by angle $\theta$
around axis $\vec{a}$:

$$\text{RRF}(\vec{v}, \theta, \vec{a}) =
\vec{v}\cos\theta
+ (\vec{a} \cdot \vec{v})\,\vec{a}\,(1 - \cos\theta)
+ (\vec{a} \times \vec{v})\sin\theta$$

Applied to update the normal and binormal:

$$\vec{x}_i = \text{RRF}(\vec{x}_{i-1},\, \theta_i,\, \vec{a}_i)
\qquad
\vec{y}_i = \text{RRF}(\vec{y}_{i-1},\, \theta_i,\, \vec{a}_i)$$

**Why this guarantees no rotation bias:** the rotation is the *minimal* rotation that
maps $\vec{z}_{i-1}$ to $\vec{z}_i$, so $\vec{x}_i$ and $\vec{y}_i$ are transported
without any additional twist.

#### Transformation matrix

At each point $\gamma(i) = (x(i), y(i), z(i))^\top$ on the spline-smoothed
centerline, the **4×4 homogeneous transformation** from the local frame
$e_i = (\vec{x}_i, \vec{y}_i, \vec{z}_i)$ to the raw (voxel) coordinate system is:

$$T_i = \begin{pmatrix}
x_{i,0} & y_{i,0} & z_{i,0} & x(i) \\
x_{i,1} & y_{i,1} & z_{i,1} & y(i) \\
x_{i,2} & y_{i,2} & z_{i,2} & z(i) \\
0 & 0 & 0 & 1
\end{pmatrix}$$

where subscript notation $x_{i,k}$ denotes the $k$-th component of $\vec{x}_i$.

#### Cross-section sampling

For a patch of half-width $W$ (in voxels), the local coordinate grid is:

$$C_i^{\text{local}} = \left\{\begin{pmatrix}u \\ v \\ 0 \\ 1\end{pmatrix}
\;:\; u, v \in \{-W, \ldots, W\}\right\}
\qquad (2W+1)^2 \text{ points}$$

Mapped to raw voxel coordinates:

$$C_i^{\text{raw}} = T_i \cdot C_i^{\text{local}}$$

The intensity (or label) at each point is obtained by tri-linear interpolation of
the image volume $I$:

$$f_i = I\bigl(C_i^{\text{raw}}\bigr)$$

The straightened volume is:

$$F = [f_0,\; f_1,\; \ldots,\; f_L]$$

---
### Time complexity comparison

| Frame | Requires | Complexity |
|-------|----------|------------|
| Frenet | $\gamma''$ | $O(L)$ but numerically unstable |
| Bishop | differential equations | $O(L)$ |
| **Zhou (SATO)** | vector rotation only | $O(L)$ — no restrictions on curve shape |

---
## Step 4 — SATO Straightening

Now we apply the swept frame to straighten both the raw CT image and the colon mask.

In [ ]:
# The crop_radius controls the cross-section size:
#   output shape = (arc_length, 2*crop_radius+1, 2*crop_radius+1)
# Default: 5 × max_tube_radius (capped at 80 voxels)
CROP_RADIUS = None   # set to an integer to override

if not os.path.exists(STRAIGHT_IMG):
    print("[4a] Straightening CT image…")
    img_f = img_arr.astype(np.float32)
    s_img = straighten_volume(img_f, cl, crop_radius=CROP_RADIUS, is_seg=False)
    sitk.WriteImage(sitk.GetImageFromArray(s_img.astype(np.int16)), STRAIGHT_IMG)
    print(f"    Saved: {STRAIGHT_IMG}  shape={s_img.shape}")
else:
    s_img_itk = sitk.ReadImage(STRAIGHT_IMG)
    s_img = sitk.GetArrayFromImage(s_img_itk)
    print(f"Loaded existing: {STRAIGHT_IMG}  shape={s_img.shape}")

if not os.path.exists(STRAIGHT_SEG):
    print("[4b] Straightening colon mask…")
    mask_f = mask_arr.astype(np.float32)
    s_seg = straighten_volume(mask_f, cl, crop_radius=CROP_RADIUS, is_seg=True)
    sitk.WriteImage(sitk.GetImageFromArray(s_seg), STRAIGHT_SEG)
    print(f"    Saved: {STRAIGHT_SEG}  shape={s_seg.shape}")
else:
    s_seg_itk = sitk.ReadImage(STRAIGHT_SEG)
    s_seg = sitk.GetArrayFromImage(s_seg_itk)
    print(f"Loaded existing: {STRAIGHT_SEG}  shape={s_seg.shape}")

print(f"\nStraightened volume shape: {s_img.shape}")
print(f"  axis 0 = {s_img.shape[0]} slices along colon (arc length ≈ {cl['edge_length']:.0f} voxels)")
print(f"  axis 1 = axis 2 = {s_img.shape[1]} pixels cross-section")

---
## Results

### Cross-sectional slices along the straightened colon

Each row is a different position along the arc-length.  
After straightening, the colon lumen always appears centred in every slice.

In [ ]:
fig = plot_straightened(s_img, s_seg, n_slices=10,
                        title="Straightened colon — cross-sections along arc")
plt.show()

### Unrolled view (coronal projection of the straightened volume)

Projecting the straightened volume along one cross-section axis gives a
single 2-D image where the entire colon is visible end-to-end — the classic
clinical CPR view.

In [ ]:
# Maximum-intensity projection of the straightened volume along axis 2
mip_straight = s_img.max(axis=2)   # shape (L, H)

fig, ax = plt.subplots(figsize=(16, 5))
lo, hi = np.percentile(mip_straight, [1, 99])
ax.imshow(mip_straight.T, cmap="gray", aspect="auto",
          vmin=lo, vmax=hi, origin="lower")
ax.set_xlabel("Slice index (arc position)")
ax.set_ylabel("Cross-section position")
ax.set_title("Unrolled colon — MIP of straightened CT (L axis = 0, cross-section = 1)",
             fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Same for the mask
mip_seg = s_seg.max(axis=2)

fig, ax = plt.subplots(figsize=(16, 3))
ax.imshow(mip_seg.T, cmap="Greens", aspect="auto", origin="lower")
ax.set_xlabel("Slice index")
ax.set_title("Unrolled colon mask", fontweight="bold")
plt.tight_layout()
plt.show()

### Pipeline summary

In [ ]:
fig = plot_pipeline_summary(img_arr, mask_arr, cl, s_img, s_seg, spacing=sp_zyx)
plt.show()

---
## Reproducibility Assessment

| Component | Status | Notes |
|-----------|--------|-------|
| **Core algorithm** (Zhou's Swept Frame) | ✅ Implemented | `straighten_volume()` in `pipeline.py` |
| **Image straightening** | ✅ Implemented | tri-linear interpolation |
| **Mask straightening** | ✅ Implemented | nearest-neighbour + hole-fill |
| **Colon segmentation** | ✅ Integrated | via TotalSegmentator 2.x |
| **Centerline extraction** | ✅ Implemented | skeletonize + BFS |
| **Multi-structure batch** | ✅ Available | `repo/straighten/parallel_straighten.py` |
| **Paper experiments** (all 8 objects) | ❌ Not reproduced | Requires original datasets |
| **Downstream analysis** (stenosis, aneurysm) | ❌ Not included | Not released by authors |

**Key dependency risks:**
- Centerline quality depends on segmentation quality (TotalSegmentator with `--fast`
  uses a 3 mm resolution model; missed segments lead to truncated centerlines).
- The skeletonization can produce spurious branches at haustral folds and
  tight curves — the longest-path BFS suppresses most, but manual inspection
  is recommended.
- Physical voxel spacing is not encoded in the straightened output; downstream
  analysis should re-derive spacing from the centerline arc-length and the
  original CT spacing.

---
## Exercises

1. **Effect of crop radius**: re-run `straighten_volume` with `crop_radius=20` and `crop_radius=60`.
   What happens to the cross-section when the radius is too small?

2. **Segmentation quality**: run TotalSegmentator without `--fast` (`fast=False`).
   Does the colon mask change? Does the centerline change?

3. **Branch pruning**: modify `compute_centerline` to prune skeleton branches shorter
   than 20 voxels before the BFS. Does this improve the path?

4. **Frenet vs Zhou**: implement the Frenet frame for this centerline.  
   What happens at the straight segments? Can you observe the rotation bias?

5. **Physical arc length**: compute the colon length in mm using the centerline
   coordinates and voxel spacing. Compare with reported clinical colon lengths
   (~150 cm).